In [ ]:
import numpy as np
import pandas as pd
import joblib
import os

from scipy.stats import randint, uniform, norm, loguniform

from sklearn.metrics import classification_report

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV

from config import SEED, N_MELS

# Entrenamiento de Random Forest

Entrenamos un RF para cada tipo de procesamiento aplicado a ciclos/ventanas.

También se probaron otros ensembles como ``GradientBoostingClassifier`` o ``HistGradientBoostingClassifier``, sin embargo, el rendimiento fue similar.

Elegimos un dataset de ciclos respiratorios o ventanas temporales

In [ ]:
dataset = 'ciclos/aug'

## Espectrogramas Mel

Cargamos las matrices guardadas en el path correspondiente

In [ ]:
train_data = np.load(f'./data_procesada/{dataset}/train_melspectrogram.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./data_procesada/{dataset}/test_melspectrogram.npz')
X_test = test_data['X']
y_test = test_data['y']

In [4]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((20724, 47616), (20724,), (5184, 47616), (5184,))

### Eleccion de hiperparámetros

Idealmente usamos `GridSearchCV` y/o `RandomizedSearchCV` para probar distintas convinaciones de hiperparametros sin sobreajustar a los datos de entrenamiento.

Nuestra métrica principal a medir es el área bajo la curva ROC.

El único hiperparámetro fijo es ``max_features='sqrt'``, tenemos demasiados features y sería computacionalmente costoso que el modelo considere todos en cada paso. Además, `max_features='log2` tuvo un peor desempeño que 'sqrt'. 

In [5]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features='sqrt')

param_distributions = {
    'max_depth': [10, 20, 30],
    'min_samples_split': [20, 40, 60],
    'min_samples_leaf': [10, 20, 30],
}

In [6]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END max_depth=10, min_samples_leaf=30, min_samples_split=60;, score=0.701 total time= 1.5min
[CV 2/5] END max_depth=10, min_samples_leaf=30, min_samples_split=60;, score=0.692 total time= 1.4min
[CV 3/5] END max_depth=10, min_samples_leaf=30, min_samples_split=60;, score=0.701 total time= 1.5min
[CV 4/5] END max_depth=10, min_samples_leaf=30, min_samples_split=60;, score=0.696 total time= 1.6min
[CV 5/5] END max_depth=10, min_samples_leaf=30, min_samples_split=60;, score=0.696 total time= 1.5min
[CV 1/5] END max_depth=20, min_samples_leaf=20, min_samples_split=40;, score=0.715 total time= 1.9min
[CV 2/5] END max_depth=20, min_samples_leaf=20, min_samples_split=40;, score=0.709 total time= 1.9min
[CV 3/5] END max_depth=20, min_samples_leaf=20, min_samples_split=40;, score=0.715 total time= 1.9min
[CV 4/5] END max_depth=20, min_samples_leaf=20, min_samples_split=40;, score=0.716 total time= 1.9min
[CV 5/5] END max_dept

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': [10, 20, ...], 'min_samples_leaf': [10, 20, ...], 'min_samples_split': [20, 40, ...]}"
,n_iter,10
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


Vemos los resultados de la validación cruzada

In [7]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
2,20,10,20,0.719193
1,20,20,40,0.714039
8,20,20,20,0.714039
3,30,20,20,0.713939
5,20,10,60,0.711439
9,30,30,20,0.705773
6,20,30,40,0.705636
7,20,30,60,0.705636
4,10,10,20,0.703212
0,10,30,60,0.697213


Elegimos y, de ser necesario, entrenamos el modelo. Si elegimos el mejor modelo del CV lo podemos escoger ya entrenado.

In [ ]:
best_model = rnd_search.best_estimator_

### Evaluación rápida

In [13]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.64      0.61      0.62      2652
           1       0.61      0.64      0.63      2532

    accuracy                           0.62      5184
   macro avg       0.62      0.62      0.62      5184
weighted avg       0.63      0.62      0.62      5184



Y lo guardamos en la carpeta correspondiente

In [ ]:
os.makedirs(f'./modelos_clasicos/modelos/{dataset}', exist_ok=True)

joblib.dump(best_model, f'./modelos_clasicos/modelos/{dataset}/melspec_rf.pkl')

## Atributos de Audio

Cargamos los features guardados en la carpeta del dataset elegido

In [ ]:
train_data = np.load(f'./data_procesada/{dataset}/train_features.npz')
X_train = train_data['X']
y_train = train_data['y']

test_data = np.load(f'./data_procesada/{dataset}/test_features.npz')
X_test = test_data['X']
y_test = test_data['y']

In [16]:
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((20724, 46), (20724,), (5184, 46), (5184,))

### Eleccion de hiperparámetros

Igual que antes, usamos `GridSearchCV` y/o `RandomizedSearchCV` para probar distintas convinaciones de hiperparametros sin sobreajustar a los datos de entrenamiento.

Nuestra métrica principal a medir es el área bajo la curva ROC.

El único hiperparámetro fijo es `max_features=None` ya que en este caso no tenemos una gran cantidad de features y se probó que funciona mejor ``'sqrt'`` y ``'log2'``

In [ ]:
rf = RandomForestClassifier(random_state=SEED, n_jobs=-1, max_features=None)

param_distributions = {
    'max_depth': [10, 20, 30],
    'min_samples_split': [10, 20, 30],
    'min_samples_leaf': [5, 10, 15]
}

In [18]:
rnd_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    verbose=3,
    random_state=SEED
)

rnd_search.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV 1/5] END max_depth=10, min_samples_leaf=15, min_samples_split=30;, score=0.668 total time=   9.0s
[CV 2/5] END max_depth=10, min_samples_leaf=15, min_samples_split=30;, score=0.672 total time=  13.0s
[CV 3/5] END max_depth=10, min_samples_leaf=15, min_samples_split=30;, score=0.682 total time=  13.6s
[CV 4/5] END max_depth=10, min_samples_leaf=15, min_samples_split=30;, score=0.669 total time=  13.5s
[CV 5/5] END max_depth=10, min_samples_leaf=15, min_samples_split=30;, score=0.683 total time=  13.9s
[CV 1/5] END max_depth=20, min_samples_leaf=10, min_samples_split=20;, score=0.681 total time=  20.0s
[CV 2/5] END max_depth=20, min_samples_leaf=10, min_samples_split=20;, score=0.688 total time=  19.3s
[CV 3/5] END max_depth=20, min_samples_leaf=10, min_samples_split=20;, score=0.693 total time=  20.3s
[CV 4/5] END max_depth=20, min_samples_leaf=10, min_samples_split=20;, score=0.686 total time=  20.2s
[CV 5/5] END max_dept

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': [10, 20, ...], 'min_samples_leaf': [5, 10, ...], 'min_samples_split': [10, 20, ...]}"
,n_iter,10
,scoring,'roc_auc'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


Observamos los resultados

In [19]:
pd.DataFrame(rnd_search.cv_results_)[['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split', 'mean_test_score']].sort_values(by='mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_score
2,20,5,10,0.689441
3,30,10,10,0.688583
8,20,10,10,0.688069
1,20,10,20,0.688069
5,20,5,30,0.687000
9,30,15,10,0.686204
6,20,15,20,0.685987
7,20,15,30,0.685987
4,10,5,10,0.675847
0,10,15,30,0.674718


Nos quedamos con un modelo

In [20]:
print("Best params:", rnd_search.best_params_)
print("Best CV score:", rnd_search.best_score_)

best_model = rnd_search.best_estimator_

Best params: {'min_samples_split': 10, 'min_samples_leaf': 5, 'max_depth': 20}
Best CV score: 0.6894407535464451


### Evaluación rápida

In [22]:
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.64      0.63      0.63      2652
           1       0.62      0.63      0.62      2532

    accuracy                           0.63      5184
   macro avg       0.63      0.63      0.63      5184
weighted avg       0.63      0.63      0.63      5184



Y guardamos el modelo

In [ ]:
joblib.dump(best_model, f'./modelos_clasicos/modelos/{dataset}/features_rf.pkl')